# Augmented classifier: training, testing, running

Same pipeline as `baseline.ipynb` — fine-tunes Ultralytics YOLO
classification (`yolo26n-cls.pt`) on SID-Set for real vs
AI-generated/tampered — but trains on **realistically degraded** images
instead of clean ones.

It still drives everything through the shared classes in
`packages/models/normal_classifier`:

- **Training + testing** — `NormalClassifierTrainer`
  (`shared_types.TrainableModel`: `.train()` / `.evaluate()` / `.save()` / `.load()`).
- **Running (inference)** — `NormalClassifierDetector`
  (`shared_types.EnsembleDetector.predict()`), the same "ready" contract
  `apps/web` consumes.

The only difference from the baseline is the data source:
`data.dataset_builder.augmented_sid_dataset()`, a re-iterable stream over
SID-Set that

- pulls a balanced subset from Hugging Face **without downloading the whole
  dataset** — one decoded image is in memory at a time;
- runs every training image through the data package's `ImageAugmenter`
  (JPEG compression, blur, resize, noise, colour jitter, centre crop), so
  the model learns to see through the degradation real uploads pick up;
- re-streams on each iteration, so one object backs `train()`'s validation
  pass, `evaluate()` and the inference demo (seed-deterministic — every
  pass sees the same images).

The held-out set is streamed **clean** (`augment=False`) so metrics measure
generalisation rather than robustness to a fixed corruption; flip it to
`augment=True` if you want the latter.


## 1. Setup — clone the repo, install deps, wire up imports

In [ ]:
%cd /content/
!git clone https://github.com/Zhongbob/TikTokTechJam2026.git

In [ ]:
%cd /content/TikTokTechJam2026
!git switch "setup"
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os,sys
os.environ["PATH"] += f":{os.path.expanduser('~/.local/bin')}"
os.environ["UV_PROJECT_ENVIRONMENT"] = sys.prefix
%cd /content/TikTokTechJam2026/packages/models/normal_classifier
!uv sync


In [ ]:
!uv pip install --system ../../data
# Might need to install local packages manually if the above doesnt work

# RESTART SESSION UNDER RUNTIME AFTER COMPLETING THE ABOVE

In [ ]:
# Run this AFTER restarting the session (cell above). Locates the .venv
# `uv sync` created and bridges its site-packages into this fresh kernel
# process via site.addsitedir() -- the API that actually processes uv's
# editable-install ".pth" redirects; a plain sys.path.insert() does not.
import glob
import importlib
import site
from pathlib import Path

candidates = [
    Path("/content/TikTokTechJam2026/.venv"),
    Path("/content/TikTokTechJam2026/packages/models/normal_classifier/.venv"),
]
venv_dir = next((c for c in candidates if c.is_dir()), None)
if venv_dir is None:
    found = glob.glob("/content/TikTokTechJam2026/**/.venv", recursive=True)
    assert found, "no .venv found under the repo -- did `uv sync` in the cell above actually succeed?"
    venv_dir = Path(found[0])

site_packages = next(venv_dir.glob("lib/python*/site-packages"))
print(f"bridging {site_packages}")
site.addsitedir(str(site_packages))

importlib.invalidate_caches()
importlib.import_module("normal_classifier")
print("'normal_classifier' imported OK")


## 2. Stream augmented training + clean validation data from SID-Set

In [ ]:
from data.dataset_builder import augmented_sid_dataset

# Both are re-iterable, memory-bounded streams -- nothing holds more than one
# decoded image at a time, and each `for sample in ...` opens a fresh Hugging
# Face stream (seed-deterministic, so every pass matches).

# Training stream: every image is run through the data package's
# ImageAugmenter (JPEG compression, blur, resize, noise, colour jitter,
# centre crop). num_augmentations controls how many are *chained* onto each
# image:
#   6      -> all six, every image (only order + parameters vary)
#   3      -> a random 3 of the 6 per image
#   (2, 5) -> a random count between 2 and 5 per image  <-- used here, for
#             realistic variety in both which corruptions and how many
train_samples = augmented_sid_dataset(
    images_per_label=1000, split="train",
    num_augmentations=(2, 5), output_size=(224, 224),
)

# Validation stream: same source, left clean (augment=False). Re-iterable, so
# train()'s val pass, evaluate() and the inference demo each re-stream it.
val_samples = augmented_sid_dataset(
    images_per_label=200, split="validation",
    augment=False, output_size=(224, 224),
)

print(f"train: {train_samples}  (~{len(train_samples)} samples)")
print(f"val:   {val_samples}  (~{len(val_samples)} samples)")


## 3. Train

`NormalClassifierTrainer.train()` exports `train_samples`/`val_samples` into
the `real/`, `ai_generated/` class-folder layout Ultralytics' classification
trainer expects, then fine-tunes `yolo26n-cls.pt` on them. Each source image
is augmented as it is streamed, then written once and reused across epochs.

In [ ]:
from normal_classifier import NormalClassifierTrainer

trainer = NormalClassifierTrainer(base_weights="yolo26n-cls.pt", image_size=224)
result = trainer.train(
    train_samples,
    val_samples=val_samples,
    output_dir="SID_YOLO_AUG",
    epochs=100,
    batch=32,
    patience=10,
    device="cpu",
    plots=True,
)
result


## 4. Test

The "testing" stage — score the trained model against a held-out set via
`.evaluate()`. SID-Set only exposes train/validation splits, so this
re-iterates `val_samples` (which re-streams from Hugging Face, still clean);
swap in a separate held-out set here if you have one.


In [ ]:
metrics = trainer.evaluate(val_samples, output_dir="SID_YOLO_AUG_eval")
print("Held-out evaluation metrics:", metrics)


In [ ]:
trainer.save("normal_classifier_augmented.pt")
print("Saved checkpoint to normal_classifier_augmented.pt")


## 5. Run (inference)

The "running" stage — `NormalClassifierDetector` wraps the saved checkpoint
and implements the same `EnsembleDetector` contract `apps/web` consumes, so
this class can be dropped straight into
`apps/web/src/web/services/factory.py`'s `get_detector()` once ready.

In [ ]:
import itertools

from normal_classifier import NormalClassifierDetector

detector = NormalClassifierDetector.from_checkpoint("normal_classifier_augmented.pt")

# val_samples re-streams on iteration; islice pulls just the first 5, so only
# those 5 images are ever decoded here.
for sample in itertools.islice(val_samples, 5):
    detection = detector.predict(sample.image)
    print(
        f"true={sample.metadata['label_name']:<10} "
        f"predicted={detection.verdict:<12} "
        f"p(ai_generated)={detection.ai_generated_probability:.2f}"
    )
